# WORKSHOP 2: Object-detection-using-web-camera
## NAME: HARISHBALA J
## REGISTER NUMBER: 212224223002

In [ ]:
import cv2
import numpy as np

# ---------------------------
# Load YOLOv4
# ---------------------------
net = cv2.dnn.readNet("yolov4.weights", "yolov4.cfg")

with open("coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

layer_names = net.getLayerNames()
output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers().flatten()]

print("YOLO loaded successfully")

# ---------------------------
# Open Webcam
# ---------------------------
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("Camera not working")
    exit()

print("Camera opened successfully")

window_name = "YOLOv4 Object Detection"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

try:
    while True:

        # ✅ If window manually closed → break loop
        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame")
            break

        height, width, _ = frame.shape

        blob = cv2.dnn.blobFromImage(
            frame,
            1/255.0,
            (320, 320),
            swapRB=True,
            crop=False
        )

        net.setInput(blob)
        outputs = net.forward(output_layers)

        boxes = []
        confidences = []
        class_ids = []

        for output in outputs:
            for detection in output:

                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]

                if confidence > 0.5:

                    center_x = int(detection[0] * width)
                    center_y = int(detection[1] * height)

                    w = int(detection[2] * width)
                    h = int(detection[3] * height)

                    x = int(center_x - w / 2)
                    y = int(center_y - h / 2)

                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)

        if len(boxes) > 0:
            indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

            if len(indexes) > 0:
                for i in indexes.flatten():
                    x, y, w, h = boxes[i]
                    label = classes[class_ids[i]]
                    conf = confidences[i]

                    cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
                    cv2.putText(
                        frame,
                        f"{label} {conf:.2f}",
                        (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0,255,0),
                        2
                    )

        cv2.imshow(window_name, frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

except KeyboardInterrupt:
    print("Stopped by user")

# ---------------------------
# Proper Close
# ---------------------------
cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

print("Camera closed properly")

YOLO loaded successfully
Camera opened successfully
